# 04. GhanaTok Training Corpus & Upgrades Preparation
Prepares the combined Twi and Ewe training corpus and sets up custom tokenizer upgrades: a targeted seed vocabulary and a dialect-aware pre-tokenizer regex for Ghanaian orthographies.

## 1. Vocabulary Size Decision (8,000 Tokens)

> **Design Decision**:
> A vocabulary size of 8,000 for GhanaTok was chosen because our combined Twi and Ewe training corpus is in the tens of thousands of sentences, not millions. A much larger vocabulary (for example 32,000, matching XLM-R) would be mostly empty slots never seen during training, and would risk the tokenizer memorising rare training sentences rather than learning genuinely useful Twi and Ewe subword units.

## 2. Upgrades: Seed Vocabulary & Dialect-Aware Pre-Tokenizer

### A. Seed Morpheme Vocabulary
BPE builds its vocabulary greedily based on frequency counts. To prevent critical Ghanaian affixes, noun class prefixes, nominal suffixes, and common high-frequency roots from being overlooked due to corpus noise or accidental character chunking, we inject a curated seed vocabulary (`SEED_MORPHEMES`).

### B. Dialect-Aware Pre-Tokenizer Pattern
Twi and Ewe use extra letters beyond the standard Latin alphabet (`ɛ`, `ɔ`, `ɖ`, `ŋ`, `ʋ`, `ƒ`) plus combining tone marks. This pattern makes sure the tokenizer treats them as normal letters, not as symbols to split on, and keeps the space before a word attached to it, matching standard byte-level BPE convention.

In [ ]:
import sys
sys.path.append("..")
from src.train_ghanatok import SEED_MORPHEMES, GHANA_PRETOKENIZER_PATTERN

print("Seed morphemes:", SEED_MORPHEMES)
print("Pre-tokenizer regex pattern:", repr(GHANA_PRETOKENIZER_PATTERN))

## 3. Build the Combined Training Corpus
Combines the Twi and Ewe training splits strictly from `train.txt`. Validation sets (`val.txt`) and test sets (`test.txt`) are excluded to prevent data leakage.

In [ ]:
from src.train_ghanatok import build_training_corpus

corpus_path = build_training_corpus()
with open(corpus_path, encoding="utf-8") as f:
    lines = f.readlines()

print("Total training lines:", len(lines))
print("\nFirst 3 lines:")
for line in lines[:3]:
    print(" ", line.strip())